In [1]:

import torch
import torch.nn.functional as F

from maskseg_build_everything import (
    build_maskvar_flex_v2,
    build_maskvar_flex,
    build_var_image_encoder,
    build_prompt_encoder,
    build_sam_image_encoder,
    build_maskvar_flex_mobile_5_stages,
)

/home/clc/miniconda3/envs/var_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/clc/miniconda3/envs/var_v2/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/clc/miniconda3/envs/var_v2/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/data/clc/maskseg/models/tinyvit.py:656: UserWarning: Overwriting tiny_vit_5m_224 in registry with models.tinyvit.tiny_vit_5m_224. This is becaus

In [2]:
device = 'cuda:3'

In [3]:
def count_parameters(model):
    """计算PyTorch模型的总参数量
    
    Args:
        model (nn.Module): PyTorch模型
        
    Returns:
        int: 总参数量
    """
    return sum(p.numel() for p in model.parameters())

In [4]:
def detailed_parameter_count(model, unit='M', print_total=True):
    """详细显示各子模块参数量"""
    def print_number(num, unit):
        if unit == 'M':
            return f'{num/1024**2:.2f}M'
        elif unit == 'K':
            return f'{num/1024:.2f}K'
        elif unit == 'B':
            return f'{num/1024**3:.2f}B'
        else:
            return f'{num:.2f}'
    
    total_params = 0
    print(f"{'Module':<30} {'Parameters':<15}")
    print("-" * 45)
    
    for name, module in model.named_children():
        module_params = sum(p.numel() for p in module.parameters())
        total_params += module_params
        print(f"{name:<30} {print_number(module_params, unit):<15}")
    
    print("-" * 45)
    print(f"{'Total':<30} {print_number(total_params, unit):<15}")
    return total_params
    

In [5]:
vqvae, maskvar, sam_image_encoder = build_maskvar_flex_mobile_5_stages('out_vqvae_5_stages_v1/ckpt/vqvae_single_epoch_50.pth', 'ckpt/mobile_sam.pt', device=device) 

print(f'vqvae params: {count_parameters(vqvae)/1024**2:.2f}M')
print(f'maskvar params: {count_parameters(maskvar)/1024**2:.2f}M')
print(f'sam_image_encoder params: {count_parameters(sam_image_encoder)/1024**2:.2f}M')


[constructor]  ==== (fusing_add_ln=0/4, fusing_mlp=0/4) ==== 
    [VAR config ] embed_dim=256, num_heads=4, depth=4, mlp_ratio=2
    [drop ratios ] drop_rate=0.1, drop_path_rate=0.1 (tensor([0.0000, 0.0333, 0.0667, 0.1000]))

vqvae params: 59.87M
maskvar params: 11.95M
sam_image_encoder params: 5.78M


In [6]:
detailed_parameter_count(maskvar)

Module                         Parameters     
---------------------------------------------
image_encoder                  2.81M          
prompt_encoder                 0.01M          
word_embed                     0.01M          
class_emb                      0.00M          
lvl_embed                      0.00M          
shared_ada_lin                 0.00M          
blocks                         7.52M          
head_nm                        0.13M          
head                           1.00M          
---------------------------------------------
Total                          11.48M         


12040012

In [7]:
detailed_parameter_count(vqvae)

Module                         Parameters     
---------------------------------------------
encoder                        24.14M         
decoder                        35.55M         
quantize                       0.16M          
quant_conv                     0.01M          
post_quant_conv                0.01M          
---------------------------------------------
Total                          59.87M         


62779233